In [3]:
from neo4j import GraphDatabase
import json
import os
import random
from dotenv import load_dotenv
import pandas as pd

load_dotenv(override=True)

True

In [4]:
URI = os.getenv('NEO4J_CONNECTION_URI')
AUTH = (os.getenv('NEO4J_USERNAME'), os.getenv('NEO4J_PASSWORD'))

In [5]:
try:
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            result = session.run("RETURN 'That is right.' AS message")
            record = result.single()
            if record:
                print(record["message"])
            else:
                print("Query returned no result.")
except Exception as e:
    print(f"Connection or query failed: {e}")

That is right.


In [67]:
def run_query(query: str, return_df: bool=False, md_path: str=''):
    try:
        with GraphDatabase.driver(URI, auth=AUTH) as driver:
            driver.verify_connectivity()

            records, summary, keys = driver.execute_query(query, database_="neo4j")
                    
    except Exception as e:
        print(f"❌ An error occurred: {e}")
    
    df = None
    data = [rec.data() for rec in records] if records else []
    if md_path:
        df = pd.DataFrame(data)
        df.to_markdown(md_path, index=False)
    if return_df:
        return pd.DataFrame(data)
    return data
    

In [10]:
query = """
MATCH (p:Paper)
WITH count(p) AS totalPapers

MATCH (paper:Paper)-[:RELATED_TO]->(t:Topic)

WITH totalPapers, t, COUNT(paper) AS papersPerTopic
where papersPerTopic > 100
RETURN t.name AS topic, 
       papersPerTopic,
       toFloat(papersPerTopic) / toFloat(totalPapers) AS ratio
ORDER BY ratio DESC
"""

topics = [t.get('topic') for t in run_query(query)]

In [11]:
topics

['Robotics and Sensor-Based Localization',
 'Advanced Neural Network Applications',
 'Autonomous Vehicle Technology and Safety',
 'Robotic Path Planning Algorithms',
 'Advanced Vision and Imaging',
 'Video Surveillance and Tracking Methods',
 'Advanced Image and Video Retrieval Techniques',
 '3D Surveying and Cultural Heritage',
 'Remote Sensing and LiDAR Applications',
 'Indoor and Outdoor Localization Technologies',
 'Traffic control and management',
 '3D Shape Modeling and Analysis',
 'Domain Adaptation and Few-Shot Learning',
 'Advanced Optical Sensing Technologies',
 'Optical measurement and interference techniques',
 'Reinforcement Learning in Robotics',
 'Human Pose and Action Recognition',
 'Anomaly Detection Techniques and Applications',
 'Target Tracking and Data Fusion in Sensor Networks',
 'Vehicle Dynamics and Control Systems',
 'Traffic Prediction and Management Techniques',
 'Adversarial Robustness in Machine Learning',
 'Analysis of Traffic Safety and Driver Behavior',


In [74]:
cat_queries_dict = {}

papers_query = {}

papers_query['MostCitedPapers_Since2020'] = """
MATCH (p:Paper)-[:RELATED_TO]->(t:Topic {{name: '{topic}'}})
where p.publicationYear > 2020
with p
match (p)-[r:PUBLISHED_IN]->(s:Source)
RETURN split(p.id, "/")[-1] AS ID, p.title AS paperTitle, p.publicationYear AS year, s.name AS publisher,  p.numberOfCitations AS citationCount
ORDER BY citationCount DESC
LIMIT 20
"""

papers_query['MostCitedPapers_Since2023'] = """
MATCH (p:Paper)-[:RELATED_TO]->(t:Topic {{name: '{topic}'}})
where p.publicationYear > 2023
with p
match (p)-[r:PUBLISHED_IN]->(s:Source)
RETURN split(p.id, "/")[-1] AS ID, p.title AS paperTitle, p.publicationYear AS year, s.name AS publisher,  p.numberOfCitations AS citationCount
ORDER BY citationCount DESC
LIMIT 20
"""

cat_queries_dict['papers'] = papers_query


publishers_query = {}

publishers_query['TopJournals_Since2020'] = """
MATCH (s:Source)<-[:PUBLISHED_IN]- (p:Paper)-[:RELATED_TO]->(t:Topic {{name: '{topic}'}})
WHERE p.publicationYear > 2020 and p.crossrefType =  "journal-article"
WITH s, p
RETURN split(s.id, "/")[-1] AS ID, 
       s.name AS publisher, 
       count(p) AS publishedCount, 
       avg(p.numberOfCitations) AS avgCitation
ORDER BY publishedCount DESC
LIMIT 20
"""

publishers_query['TopJournals_Since2023'] = """
MATCH (s:Source)<-[:PUBLISHED_IN]- (p:Paper)-[:RELATED_TO]->(t:Topic {{name: '{topic}'}})
WHERE p.publicationYear > 2023 and p.crossrefType =  "journal-article"
WITH s, p
RETURN split(s.id, "/")[-1] AS ID, 
       s.name AS publisher, 
       count(p) AS publishedCount, 
       avg(p.numberOfCitations) AS avgCitation
ORDER BY publishedCount DESC
LIMIT 20
"""

publishers_query['TopConference_Since2020'] = """
MATCH (s:Source)<-[:PUBLISHED_IN]- (p:Paper)-[:RELATED_TO]->(t:Topic {{name: '{topic}'}})
WHERE p.publicationYear > 2020 and p.crossrefType =  "proceedings-article"
WITH s, p
RETURN split(s.id, "/")[-1] AS ID, 
       s.name AS publisher, 
       count(p) AS publishedCount, 
       avg(p.numberOfCitations) AS avgCitation
ORDER BY publishedCount DESC
LIMIT 20
"""

publishers_query['TopConference_Since2023'] = """
MATCH (s:Source)<-[:PUBLISHED_IN]- (p:Paper)-[:RELATED_TO]->(t:Topic {{name: '{topic}'}})
WHERE p.publicationYear > 2023 and p.crossrefType =  "proceedings-article"
WITH s, p
RETURN split(s.id, "/")[-1] AS ID, 
       s.name AS publisher, 
       count(p) AS publishedCount, 
       avg(p.numberOfCitations) AS avgCitation
ORDER BY publishedCount DESC
LIMIT 20
"""

publishers_query['TopPublishers_InTopic'] = """
MATCH (s:Source)-[:RELATED_TO]->(t:Topic {{name: '{topic}'}})
RETURN split(s.id, "/")[-1] AS ID, 
       s.name AS publisher, 
       s.hIndex AS hIndex
ORDER BY hIndex DESC
LIMIT 50
"""

cat_queries_dict['publishers'] = publishers_query

researchers_query = {}

researchers_query['TopResearchers_Papers&Topic'] = """
MATCH (t1:Topic {{name: '{topic}'}}) <-[r1:RELATED_TO]- (r:Researcher) <-[:AUTHORED_BY]- (p:Paper) -[:RELATED_TO]-> (t:Topic {{name: '{topic}'}})
WHERE p.publicationYear > 2020 AND r1.count > 10
with r, r1, p
match (r) -[:LAST_KNOWN_AFFILIATION]-> (i:Institution)
RETURN DISTINCT split(r.id, "/")[-1] AS ID, r.name AS name, r.hIndex as hIndex , i.name As lastInstitution, count(p) AS numberOfWork
ORDER BY numberOfWork DESC
LIMIT 50
"""

researchers_query['TopResearchers_Topic'] = """
MATCH (t1:Topic {{name: '{topic}'}}) <-[r1:RELATED_TO]- (r:Researcher)
WHERE r1.count > 10
with r, r1
match (r) -[:LAST_KNOWN_AFFILIATION]-> (i:Institution)
RETURN DISTINCT split(r.id, "/")[-1] AS ID, r.name AS name, r.hIndex as hIndex , i.name As lastInstitution, r1.count AS numberOfWorkInTopic
ORDER BY numberOfWorkInTopic DESC
LIMIT 50
"""

cat_queries_dict['researchers'] = researchers_query

institutions_query = {}

institutions_query['TopInstitutions_Researchers&Topic'] = """
match (p:Paper)-[:AUTHORED_BY]->(rr:Researcher)
WHERE p.publicationYear > 2020
with p, rr
match (rr)-[r2:AFFILIATED_FOR_WORK {{paperId:p.id}}]-> (i:Institution) -[r:RELATED_TO]->(t1:Topic {{name: '{topic}'}})
RETURN split(i.id, "/")[-1] AS ID, i.name AS name, i.countryCode as country, i.hIndex as hIndex, count(p) AS workCountInTopic
ORDER BY workCountInTopic DESC
LIMIT 50
"""

institutions_query['TopInstitutions_Topic'] = """
match (i:Institution) -[r:RELATED_TO]->(t:Topic {{name: '{topic}'}})
RETURN split(i.id, "/")[-1] AS ID, i.name AS name, i.countryCode as country, i.hIndex as hIndex, r.count AS workCountInTopic
ORDER BY hIndex DESC
LIMIT 50
"""

cat_queries_dict['institutions'] = institutions_query

funders_query = {}

funders_query['TopFunders'] = """
match (t:Topic {{name: '{topic}'}})<-[:RELATED_TO]-(p:Paper)-[:FUNDED_BY]->(f:Funder)
RETURN split(f.id, "/")[-1] AS ID, f.name AS name, f.hIndex as hIndex, count(p) as workCountInTopic
ORDER BY workCountInTopic DESC
limit 50
"""

funders_query['TopFunders_Since2020'] = """
match (t:Topic {{name: '{topic}'}})<-[:RELATED_TO]-(p:Paper)-[:FUNDED_BY]->(f:Funder)
where p.publicationYear > 2020
RETURN split(f.id, "/")[-1] AS ID, f.name AS name, f.hIndex as hIndex, count(p) as workCountInTopic
ORDER BY workCountInTopic DESC
limit 50
"""

cat_queries_dict['funders'] = funders_query

In [77]:
import time

In [78]:
insights_base_dir = 'insights'

for topic_index, topic in enumerate(topics):
    topic_dir = os.path.join(insights_base_dir, f'{topic_index+1}.'+topic)
    os.makedirs(topic_dir, exist_ok=True)

    print(topic)
    for cat, queries_dict in cat_queries_dict.items():
        cat_dir = os.path.join(topic_dir, cat)
        os.makedirs(cat_dir, exist_ok=True)

        print(f'\t{cat}')
        

        for fname, query in queries_dict.items():
            
            df = run_query(query.format(topic=topic), return_df=True, md_path=os.path.join(cat_dir, fname+'.md'))
            df.to_csv(os.path.join(cat_dir, fname+'.csv'), index=False)
            
            time.sleep(0.5)

    print()

Robotics and Sensor-Based Localization
	papers


	publishers
	researchers
	institutions
	funders

Advanced Neural Network Applications
	papers
	publishers
	researchers
	institutions
	funders

Autonomous Vehicle Technology and Safety
	papers
	publishers
	researchers
	institutions
	funders

Robotic Path Planning Algorithms
	papers
	publishers
	researchers
	institutions
	funders

Advanced Vision and Imaging
	papers
	publishers
	researchers
	institutions
	funders

Video Surveillance and Tracking Methods
	papers
	publishers
	researchers
	institutions
	funders

Advanced Image and Video Retrieval Techniques
	papers
	publishers
	researchers
	institutions
	funders

3D Surveying and Cultural Heritage
	papers
	publishers
	researchers
	institutions
	funders

Remote Sensing and LiDAR Applications
	papers
	publishers
	researchers
	institutions
	funders

Indoor and Outdoor Localization Technologies
	papers
	publishers
	researchers
	institutions
	funders

Traffic control and management
	papers
	publishers
	researchers
	institutions
	funders

3D Shap